# Open-source LLM risk judges — T-MDP sandbox (experimental)

Runs a set of **open-source instruct models** as the security risk judge over the same
**542 frozen prompts** the Claude judge was calibrated on, and writes one
`responses_<model>.jsonl` per model in the exact `LLMJudge` cache schema
(`{key, model, p_malicious, rationale, raw_response}`, `key = sha256(model + "\0" + prompt)`).

Back on your machine, score each file with **zero live model calls**:
```bash
/Users/ethandoherty/tmdp-sandbox/.venv/bin/python runs/run_llm_judge_calibration.py \
    --model "Qwen/Qwen2.5-7B-Instruct" \
    --external-responses responses_Qwen_Qwen2.5-7B-Instruct.jsonl \
    --out-dir runs/llm_judge_calibration_qwen25_7b
```

**Runtime:** use a GPU runtime (free T4 works; ~30–60 min per 7B model over 542 prompts).
The prompts are recorded public OTRF telemetry replayed offline — nothing here executes commands.

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────
PROMPTS_URL = "https://raw.githubusercontent.com/edoherty2017/tmdp-sandbox/experimental/runs/oss_judges/prompts.jsonl"

MODELS = [
    "Qwen/Qwen2.5-7B-Instruct",          # ungated
    "microsoft/Phi-3.5-mini-instruct",   # ungated
    "HuggingFaceH4/zephyr-7b-beta",      # ungated
    # Gated (accept license on HF first, set HF_TOKEN below):
    # "meta-llama/Llama-3.1-8B-Instruct",
    # "mistralai/Mistral-7B-Instruct-v0.3",
    # "google/gemma-2-9b-it",
]

HF_TOKEN = ""        # only needed for gated models
LIMIT = None          # e.g. 20 for a smoke run; None = all 542
MAX_NEW_TOKENS = 256

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes sentencepiece

In [ ]:
import hashlib, json, re, urllib.request

prompts = []
with urllib.request.urlopen(PROMPTS_URL) as fh:
    for line in fh.read().decode("utf-8").splitlines():
        if line.strip():
            prompts.append(json.loads(line))
if LIMIT:
    prompts = prompts[:LIMIT]
print(f"{len(prompts)} prompts loaded")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

JSON_RE = re.compile(r"\{[^{}]*\"p_malicious\"[^{}]*\}", re.DOTALL)

def parse_reply(text):
    """Mirror LLMJudge parsing: find the JSON object, clamp p to [0,1]."""
    m = JSON_RE.search(text)
    if not m:
        return None, "unparseable model reply"
    try:
        obj = json.loads(m.group(0))
        p = float(obj["p_malicious"])
    except (json.JSONDecodeError, KeyError, TypeError, ValueError):
        return None, "unparseable model reply"
    return max(0.0, min(1.0, p)), str(obj.get("rationale", ""))

def run_model(model_id):
    slug = model_id.replace("/", "_")
    out_path = f"responses_{slug}.jsonl"
    tok_kw = {"token": HF_TOKEN} if HF_TOKEN else {}
    tokenizer = AutoTokenizer.from_pretrained(model_id, **tok_kw)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=BitsAndBytesConfig(load_in_4bit=True),
        device_map="auto",
        **tok_kw,
    )
    n_null = 0
    with open(out_path, "w") as fh:
        for i, item in enumerate(prompts):
            msgs = [{"role": "user", "content": item["prompt"]}]
            ids = tokenizer.apply_chat_template(
                msgs, add_generation_prompt=True, return_tensors="pt"
            ).to(model.device)
            with torch.no_grad():
                out = model.generate(
                    ids, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                    pad_token_id=tokenizer.eos_token_id,
                )
            text = tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)
            p, rationale = parse_reply(text)
            if p is None:
                n_null += 1
            key = hashlib.sha256(f"{model_id}\x00{item['prompt']}".encode()).hexdigest()
            fh.write(json.dumps({
                "key": key,
                "model": model_id,
                "p_malicious": p,
                "rationale": rationale,
                "raw_response": text,
            }) + "\n")
            if (i + 1) % 25 == 0:
                print(f"  {i+1}/{len(prompts)} (nulls={n_null})")
    del model
    torch.cuda.empty_cache()
    print(f"{model_id}: done -> {out_path} ({n_null} nulls)")
    return out_path

outputs = [run_model(m) for m in MODELS]

In [ ]:
# Zip everything for download
import shutil
from google.colab import files
shutil.make_archive("oss_judge_responses", "zip", ".", base_dir=None)
files.download("oss_judge_responses.zip")